In [1]:
import numpy as np
import pandas as pd
import dagshub 
import mlflow

c:\Users\win11\OneDrive\Desktop\MLops\Project\yt_plugin\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dagshub.init(repo_name = 'yt_chrome_plugin', repo_owner = 'AmanGupta8423', mlflow = True)
mlflow.set_experiment("Experiment1-Bow-Tfidf")
mlflow.set_tracking_uri('https://dagshub.com/AmanGupta8423/yt_chrome_plugin.mlflow')


Accessing as AmanGupta8423

Initialized MLflow to track repo "AmanGupta8423/yt_chrome_plugin"

Repository AmanGupta8423/yt_chrome_plugin initialized!

2026/11/26 17:07:53 INFO mlflow.tracking.fluent: Experiment with name 'Experiment1-Bow-Tfidf' does not exist. Creating a new experiment.


In [3]:
from  sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [4]:
df = pd.read_csv('./data/raw/dataset.csv')
df.shape

(36793, 2)

In [5]:
df.isnull().sum()

clean_comment    131
category           0
dtype: int64

In [6]:
df = df.dropna(subset='clean_comment')

In [7]:
df.shape

(36662, 2)

In [8]:
def run_experiment(vectorizer_type, ngram_range, vectorizer_max_features, vectorizer_name):
    if vectorizer_type == 'BoW':
        vectorizer = CountVectorizer(ngram_range= ngram_range, max_features=vectorizer_max_features)
    elif vectorizer_type == 'Tfidf':
        vectorizer = TfidfVectorizer(ngram_range=ngram_range,max_features=vectorizer_max_features)

    X_train,X_test,y_train,y_test = train_test_split(df['clean_comment'],df['category'], test_size = 0.2, random_state=42, stratify=df['category'])

    X_train = vectorizer.fit_transform(X_train)
    X_test = vectorizer.transform(X_test)

    with mlflow.start_run():
        mlflow.set_tag('mlflow.runName', f'{vectorizer_name}_{ngram_range}_RandomForest')
        mlflow.set_tag('experiment_type','feature_engineering')
        mlflow.set_tag('model_type', 'RandomForestClassifier')
        mlflow.set_tag('description','BoW vs Tfidf and finding the ngram value too')
        
        mlflow.log_param('vectorize_type',vectorizer_type)
        mlflow.log_param('ngram_range', ngram_range)
        mlflow.log_param('vectorizer_max_features',vectorizer.max_features)
    
        n_estimators = 100
        max_depth = 10
    
        mlflow.log_param('n_estimators',n_estimators)
        mlflow.log_param('max_depth',max_depth)
    
        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth,random_state=42)
        model.fit(X_train,y_train)
    
        y_pred = model.predict(X_test)
    
        accuracy = accuracy_score(y_test,y_pred)
    
        classification_rep = classification_report(y_test,y_pred,output_dict=True)
    
        for label,metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f'{label}_{metric}',value)

        conf_matrix = confusion_matrix(y_test,y_pred)
        plt.figure(figsize=(8,6))
        sns.heatmap(conf_matrix,annot = True, fmt = 'd',cmap='Blues')
        plt.xlabel('Predicted')
        plt.ylabel('Actual')
        plt.title(f'Confusion Matrix: {vectorizer_name}, {ngram_range}')
    
        os.makedirs("./reports", exist_ok=True)

        plt.savefig('./reports/confusion_matrix.png')
        mlflow.log_artifact('./reports/confusion_matrix.png')
        plt.close()

        mlflow.sklearn.log_model(model, name="random_forest_model",skops_trusted_types=["sklearn.tree._tree.Tree"])


ngram_ranges = [(1,1),(1,2),(1,3)]
max_features = 5000

for ngram_range in ngram_ranges:
    run_experiment(vectorizer_type='BoW',ngram_range=ngram_range, vectorizer_max_features=max_features,vectorizer_name='BoW')
    run_experiment(vectorizer_type='Tfidf',ngram_range=ngram_range, vectorizer_max_features=max_features,vectorizer_name='TF-IDF')




🏃 View run BoW_(1, 1)_RandomForest at: https://dagshub.com/AmanGupta8423/yt_chrome_plugin.mlflow/#/experiments/2/runs/afcd332d0d2c458cb174d7057d56d05e
🧪 View experiment at: https://dagshub.com/AmanGupta8423/yt_chrome_plugin.mlflow/#/experiments/2
🏃 View run TF-IDF_(1, 1)_RandomForest at: https://dagshub.com/AmanGupta8423/yt_chrome_plugin.mlflow/#/experiments/2/runs/503bfb8638f64ecd82d9d1a38a2efbe3
🧪 View experiment at: https://dagshub.com/AmanGupta8423/yt_chrome_plugin.mlflow/#/experiments/2
🏃 View run BoW_(1, 2)_RandomForest at: https://dagshub.com/AmanGupta8423/yt_chrome_plugin.mlflow/#/experiments/2/runs/99018cc1356c4311a3df2f76cc482da6
🧪 View experiment at: https://dagshub.com/AmanGupta8423/yt_chrome_plugin.mlflow/#/experiments/2
🏃 View run TF-IDF_(1, 2)_RandomForest at: https://dagshub.com/AmanGupta8423/yt_chrome_plugin.mlflow/#/experiments/2/runs/699add487f82461bb60930917fc15769
🧪 View experiment at: https://dagshub.com/AmanGupta8423/yt_chrome_plugin.mlflow/#/experiments/2
🏃 View